In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from source.config import DATA_DIR, SRC_DIR

import pandas as pd
import os
import re
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm
import numpy as np
import osmnx as ox

In [20]:
file_path = DATA_DIR / "raw/cctv_info.csv"

df = pd.read_csv(file_path)

df.head()

,camera_id,location,location_detail,location_en,location_detail_en,latitude,longitude,ip_address,image_count
0,915,TF-KS-01-วงเวียนใหญ่,วงเวียนใหญ่,connect BMAcctv,connect BMAcctv,13.72639,100.49213,10.156.102.15,613
1,137,สะพานลอยใกล้บิ๊กซีเอ็กซ์ตร,รถมาจากแยกรัชดา-ลาดพร้าว-มุ่,Footbridge near Big C Extra Entrance to Soi La...,From Ratchada-Lat Phrao Intersection towards L...,13.80940,100.56754,10.130.129.126,613
2,219,สะพานลอยใกล้ซอยรัชดาภิเษก-14,รถมาจากแยกรัชดา-สุทธิสาร-มุ่,Footbridge near Soi Ratchadaphisek 14 on Ratch...,From Ratchada-Sutthisan Intersection towards H...,13.78194,100.57389,10.117.129.128,613
3,169,สะพานลอยหน้าวัดเสาหิน-ใกล้ซ,รถมาจากสะพานพระราม-7-มุ่งหน้,Footbridge in front of Wat Sao Hin Near Soi ...,From Rama7 Bridge towards Wongsawang Intersection,13.81863,100.51776,10.125.129.126,609
4,332,แยกราชวัตร,รถมุ่งหน้าแยกพิชัย,Rat Cha Wat Intersection,Towards Phi Chai Intersection,13.77951,100.52231,10.102.101.105,615


## Date Scope

In [21]:
ds_path = DATA_DIR / "raw/cctv_images"

In [22]:
pattern = r"(\d{8}_\d{6})(?=\.jpg$)"

In [ ]:
# Extract date in ds
camera_id = os.listdir(ds_path)

date_list = []

for id in camera_id:
    temp_path = ds_path / id
    for path in os.listdir(temp_path):
        #print(path)
        match = re.search(pattern , path)

        date = match.group(1).split("_")[0]
        if date not in date_list:
            date_list.append(date)

date_list

['20260825',
 '20260826',
 '20260827',
 '20260828',
 '20260829',
 '20260830',
 '20260831']

In [28]:
year_list = []
for d in date_list:
    year = d[:4]
    if year not in year_list:
        year_list.append(year)

year_list 

['2026']

## Traffic Longdo Web Scraping

In [24]:
page_url = "https://traffic.longdo.com/download"

response = requests.get(page_url, timeout= 30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

soup


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">

<html dir="ltr" lang="th" xml:lang="th" xmlns="http://www.w3.org/1999/xhtml" xmlns:fb="http://www.facebook.com/2008/fbml" xmlns:og="http://opengraphprotocol.org/schema/">
<meta content="IE=8" http-equiv="X-UA-Compatible"/>
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<link href="/sites/all/themes/traffic_theme/logo_90.png" rel="image_src"/>
<meta content="รายงานสภาพจราจร　ข้อมูลจราจร　กรุงเทพ　ประเทศไทย　ล่าสุด　Online Map with Traffic Data" name="description"/>
<meta content="map,traffic,traffic jam,road,street,route,routing,bangkok,traffic camera,cctv,event,incident,accident,construction,rain,flood,crowd,mob,exhibition,fair,แผนที่,จราจร,ถนน,เส้นทาง,แนะนำเส้นทาง,กรุงเทพ,กล้องวงจรปิด,กล้องจราจร,ข้อมูลจราจร,เหตุการณ์,อุบัติเหตุ,ก่อสร้าง,ปิดถนน,ฝนตก,น้ำท่วม,ชุมนุม,ประท้วง,จัดงาน,นิทรรศการ" name="keywords"/>
<title>
			ดาวน์โหลด | Longdo Traffic รายงานส

In [25]:
traffic_idx = soup.select('li:-soup-contains("ข้อมูลดัชนีรถติดย้อนหลัง") > ul > li')
car_accident = soup.select('li:-soup-contains("ข้อมูลเหตุการณ์ย้อนหลัง") > ul > li')

car_accident

[<li><a href="https://event.longdo.com/feed/2026">ปี 2569 (2026)</a></li>,
 <li><a href="https://event.longdo.com/feed/2025">ปี 2568 (2025)</a></li>,
 <li><a href="https://event.longdo.com/feed/2024">ปี 2567 (2024)</a></li>,
 <li><a href="https://event.longdo.com/feed/2023">ปี 2566 (2023)</a></li>,
 <li><a href="https://event.longdo.com/feed/2022">ปี 2565 (2022)</a></li>,
 <li><a href="https://event.longdo.com/feed/2021">ปี 2564 (2021)</a></li>,
 <li><a href="https://event.longdo.com/feed/2020">ปี 2563 (2020)</a></li>,
 <li><a href="https://event.longdo.com/feed/2019">ปี 2562 (2019)</a></li>,
 <li><a href="https://event.longdo.com/feed/2018">ปี 2561 (2018)</a></li>,
 <li><a href="https://event.longdo.com/feed/2017">ปี 2560 (2017)</a></li>,
 <li><a href="https://event.longdo.com/feed/2016">ปี 2559 (2016)</a></li>,
 <li><a href="https://event.longdo.com/feed/2015">ปี 2558 (2015)</a></li>,
 <li><a href="https://event.longdo.com/feed/2014">ปี 2557 (2014)</a></li>,
 <li><a href="https://eve

In [29]:
# Extract Link from element
# Car accident
car_accident_link_list = [] 
for item in car_accident:
    car_accident_link_list.append(item.select_one("a")["href"])

# Traffix Index
traffic_idx_link_list = [] 
for item in traffic_idx:
    traffic_idx_link_list.append(item.select_one("a")["href"])

traffic_idx_link_list

['/api/raw/trafficindex/2026',
 '/api/raw/trafficindex/2025',
 '/api/raw/trafficindex/2024',
 '/api/raw/trafficindex/2023',
 '/api/raw/trafficindex/2022',
 '/api/raw/trafficindex/2021',
 '/api/raw/trafficindex/2020',
 '/api/raw/trafficindex/2019',
 '/api/raw/trafficindex/2018',
 '/api/raw/trafficindex/2017',
 '/api/raw/trafficindex/2016',
 '/api/raw/trafficindex/2015',
 '/api/raw/trafficindex/2014',
 '/api/raw/trafficindex/2013',
 '/api/raw/trafficindex/2012']

In [ ]:
# Make dirs
longdo_dir = DATA_DIR / "raw/traffic_longdo"
car_acc_dir = longdo_dir / "car_accident"
traffic_idx_dir = longdo_dir / "traffix_index"

longdo_dir.mkdir(exist_ok = True, parents = True)
car_acc_dir.mkdir(exist_ok = True, parents = True)
traffic_idx_dir.mkdir(exist_ok = True, parents = True)

# Download csv file
def download_info(link_list, year_list, download_dir):
    """
    Download CSV files from a list of URLs and save them by year.

    Args:
        link_list (list[str]): List of URLs pointing to CSV files.
        year_list (list[str]): List of years used to identify and name files.
        download_dir (Path): Directory where the CSV files will be saved.

    Returns:
        None: Downloads CSV files to the specified directory.

    Raises:
        requests.exceptions.RequestException: If an HTTP request fails.
        OSError: If a file cannot be written to the directory.
    """
    for link in tqdm(link_list):
        for year in year_list:
            if year in link:
                temp_response = requests.get(link, timeout= 30)
                temp_response.raise_for_status()

                with open(download_dir / f"{year}.csv", "wb") as f:
                    f.write(temp_response.content)

        time.sleep(1)

download_info(car_accident_link_list, year_list, car_acc_dir)
download_info(traffic_idx_link_list, year_list, traffic_idx_dir)

100%|██████████| 15/15 [00:17<00:00,  1.14s/it]


## ERA5 Weather DATA

In [48]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 13.7563,
    "longitude": 100.5018,
    "start_date": "2024-01-01",
    "end_date": "2024-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        "surface_pressure"
    ],
    "timezone": "Asia/Bangkok",
    "models": "era5"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data["hourly"])

df.head()

,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,surface_pressure
0,2024-01-01T00:00,26.5,78,0.0,6.8,1011.8
1,2024-01-01T01:00,26.6,77,0.0,7.6,1011.8
2,2024-01-01T02:00,26.8,74,0.0,7.7,1011.2
3,2024-01-01T03:00,25.3,78,0.0,7.6,1010.7
4,2024-01-01T04:00,25.8,76,0.0,7.2,1010.6


## OSM API

In [ ]:
# Example CCTV camera location
lat = 13.72639
lon = 100.49213

# Get driving road network around camera
G = ox.graph_from_point(
    (lat, lon),
    dist=500,
    network_type="drive"
)

# Convert network into tables
nodes, edges = ox.graph_to_gdfs(G)

# Inspect available attributes
print(edges.columns.tolist())

# Examine recorded road attributes
columns = [
    "highway", "lanes",
    "maxspeed", "oneway"
]

edges.reindex(columns=columns).head()


['osmid', 'highway', 'lanes', 'name', 'oneway', 'reversed', 'length', 'geometry', 'maxspeed', 'bridge', 'junction']


highway lanes maxspeed  oneway
u         v          key                                    
291425517 1539171190 0        primary     4      NaN    True
          9407212769 0    residential   NaN      NaN   False
291425519 3585868138 0    residential   NaN       30   False
          291425517  0        primary     4      NaN    True
291425853 1539171183 0        primary     4      NaN    True

In [40]:
# edges = GeoDataFrame returned by ox.graph_to_gdfs(G)
road_df = edges.copy()

def first_value(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value

def parse_number(value):
    value = first_value(value)

    if pd.isna(value):
        return np.nan

    match = re.search(r"\d+(\.\d+)?", str(value))
    return float(match.group()) if match else np.nan

road_df["road_type"] = (
    road_df["highway"].apply(first_value)
)

road_df["lane_count"] = (
    road_df["lanes"].apply(parse_number)
)

road_df["speed_limit_raw"] = (
    road_df["maxspeed"]
)

road_df["oneway_feature"] = (
    road_df["oneway"].astype("boolean")
)

road_df["bridge_feature"] = (
    road_df["bridge"].notna().astype(int)
)

features = road_df[
    [
        "road_type",
        "lane_count",
        "speed_limit_raw",
        "oneway_feature",
        "bridge_feature"
    ]
]

features.head()


road_type  lane_count speed_limit_raw  \
u         v          key                                            
291425517 1539171190 0        primary         4.0             NaN   
          9407212769 0    residential         NaN             NaN   
291425519 3585868138 0    residential         NaN              30   
          291425517  0        primary         4.0             NaN   
291425853 1539171183 0        primary         4.0             NaN   

                          oneway_feature  bridge_feature  
u         v          key                                  
291425517 1539171190 0              True               0  
          9407212769 0             False               0  
291425519 3585868138 0             False               0  
          291425517  0              True               1  
291425853 1539171183 0              True               0